# 🩺 MedGemma server for Aura — Google Colab (free GPU)

Serves **MedGemma 1.5 4B (vision)** on a free Colab **T4 GPU** via **Ollama**, exposed as an
OpenAI-compatible endpoint through a free Cloudflare tunnel (no account, no token).

### How to use
1. **Runtime → Change runtime type → T4 GPU**.
2. **Runtime → Run all**. First run installs Ollama + pulls the model (~3 GB, a few minutes).
3. That's it — the endpoint is **auto-published to the app** (via ntfy), so the app picks it up
   automatically. No copy-paste, no `.env` edit needed.
4. **Keep this tab open** — the link lives only while the runtime runs.

_Uses Ollama's official `medgemma1.5` (vision-capable). First model call cold-starts (~1 min); then it's fast. The app matches by the ntfy topic in `.env.local` (`MEDGEMMA_NTFY_TOPIC`)._

In [ ]:
!nvidia-smi -L || echo 'No GPU! Runtime > Change runtime type > T4 GPU, then Run all.'

In [ ]:
import os, subprocess, time, re, requests
NTFY_TOPIC = 'aura-med-9k3f7q2x8w'   # must match MEDGEMMA_NTFY_TOPIC in the app .env.local
print('Installing zstd + Ollama (prebuilt, no compile)...')
os.system('apt-get -qq install -y zstd >/tmp/zstd.log 2>&1')
os.system('curl -fsSL https://ollama.com/install.sh | sh >/tmp/ollama_install.log 2>&1')
subprocess.Popen(['ollama', 'serve'], stdout=open('/tmp/ollama.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(8)
print('Pulling MedGemma 1.5 4B (vision) ~3GB...')
os.system('ollama pull medgemma1.5')
print('Model ready. Serving on GPU via Ollama.')
os.system('test -f cloudflared || (wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared)')
cf = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:11434', '--http-host-header', 'localhost:11434', '--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
t0 = time.time()
for line in cf.stdout:
    m = re.search(r'https://[-a-z0-9.]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0); break
    if time.time() - t0 > 90: break
endpoint = (url or 'NONE') + '/v1/chat/completions'
try:
    requests.post('https://ntfy.sh/' + NTFY_TOPIC, data=endpoint.encode())
    print('Endpoint auto-published to the app (ntfy) — nothing to paste.')
except Exception as e:
    print('ntfy publish failed:', e)
print('=' * 60)
print('MEDGEMMA_ENDPOINT=' + endpoint)
print('MEDGEMMA_MODEL=medgemma1.5')
print('=' * 60)
print('Keep this tab open. The app auto-syncs this endpoint.')

In [ ]:
# (Optional) quick self-test
import requests
r = requests.post('http://127.0.0.1:11434/v1/chat/completions',
                  json={'model': 'medgemma1.5', 'messages': [{'role': 'user', 'content': 'Reply with: MedGemma online'}], 'max_tokens': 16})
print(r.json().get('choices', [{}])[0].get('message', {}).get('content', r.text))